# Credit Card Fraud - Imbalanced Dataset

**Fraud Detection / Imbalanced Classification** — Detect fraudulent credit card transactions.

Models: CatBoost · LightGBM · XGBoost (with class weights)
Baselines: LazyPredict · FLAML AutoML
Dataset: Kaggle Credit Card Fraud (284,807 rows × 31 columns, ~0.17% fraud)
Target: `Class`

## 2 · Project Overview

We build a fraud detection system for credit card transactions. The dataset is **extremely imbalanced** (~0.17% fraud rate), making standard accuracy meaningless. We focus on **PR-AUC and recall** as primary metrics, using class weights and threshold calibration.

## 3 · Learning Objectives

1. Handle extreme class imbalance in binary classification.
2. Use PR-AUC and recall as primary metrics instead of accuracy.
3. Apply class weight calibration in boosting models.
4. Understand the precision-recall tradeoff in fraud detection.
5. Compare boosting models on a large-scale imbalanced dataset.

## 4 · Problem Statement

Given PCA-transformed transaction features and the transaction amount, predict whether a credit card transaction is **fraudulent** (Class=1) or **legitimate** (Class=0).

## 5 · Why This Project Matters

- Credit card fraud costs $28+ billion annually worldwide.
- Automated detection saves manual review effort and catches fraud faster.
- This dataset is the gold standard for imbalanced classification research.
- The extreme imbalance (~0.17%) mirrors real-world fraud rates.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 284,807 |
| **Columns** | 31 |
| **Features** | V1–V28 (PCA components), Time, Amount |
| **Target** | `Class` (1=fraud, 0=legitimate) |
| **Fraud rate** | ~0.17% (492 frauds) |
| **Source** | kagglehub: `mlg-ulb/creditcardfraud` |

## 7 · Dataset Source and License Notes

- **Source**: Worldline and Université Libre de Bruxelles (ULB).
- **Kaggle**: `mlg-ulb/creditcardfraud`
- **License**: Open Database License (ODbL).
- **Note**: Features V1–V28 are PCA-transformed for confidentiality.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install_if_missing(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg, imp in [("catboost", None), ("lightgbm", None), ("xgboost", None),
                 ("flaml", None), ("lazypredict", None), ("kagglehub", None)]:
    _install_if_missing(pkg, imp)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, re, warnings, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, average_precision_score)
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
print("Imports done.")

Imports done.


## 10 · Configuration

In [3]:
SAVE_DIR = r"E:/Github/Machine-Learning-Projects/Classification/Credit Card Fraud - Imbalanced Dataset"
os.makedirs(SAVE_DIR, exist_ok=True)
TARGET = "Class"
MAX_ROWS = 50000  # Sample to prevent timeout
print(f"Save dir: {SAVE_DIR}")

Save dir: E:/Github/Machine-Learning-Projects/Classification/Credit Card Fraud - Imbalanced Dataset


## 11 · Dataset Download and Loading

The full dataset has 284K rows. We sample to 50K for training speed while preserving all fraud cases.

In [4]:
import kagglehub
data_path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
import glob
csv_file = glob.glob(os.path.join(data_path, "*.csv"))[0]
df_full = pd.read_csv(csv_file)
print(f"Full dataset shape: {df_full.shape}")
print(f"Full fraud rate: {df_full[TARGET].mean():.6f}")

# Stratified sample: keep ALL fraud, sample from legitimate
fraud = df_full[df_full[TARGET] == 1]
legit = df_full[df_full[TARGET] == 0].sample(n=min(MAX_ROWS, len(df_full[df_full[TARGET]==0])), random_state=SEED)
df = pd.concat([fraud, legit], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"\nSampled shape: {df.shape}")
print(f"Sampled fraud rate: {df[TARGET].mean():.4f}")
print(f"Fraud cases preserved: {df[TARGET].sum()} / {fraud.shape[0]}")
del df_full  # free memory

  0%|                                              | 0.00/66.0M [00:00<?, ?B/s]

  2%|▌                                     | 1.00M/66.0M [00:01<01:41, 671kB/s]

  3%|█                                    | 2.00M/66.0M [00:01<00:49, 1.36MB/s]

  5%|█▋                                   | 3.00M/66.0M [00:01<00:30, 2.18MB/s]

  8%|██▊                                  | 5.00M/66.0M [00:02<00:15, 4.09MB/s]

...

 92%|██████████████████████████████████▏  | 61.0M/66.0M [00:05<00:00, 13.9MB/s]

 97%|███████████████████████████████████▉ | 64.0M/66.0M [00:06<00:00, 6.82MB/s]

100%|█████████████████████████████████████| 66.0M/66.0M [00:06<00:00, 10.0MB/s]

Extracting files...


Full dataset shape: (284807, 31)
Full fraud rate: 0.001727

Sampled shape: (50492, 31)
Sampled fraud rate: 0.0097
Fraud cases preserved: 492 / 492


## 12 · Data Validation Checks

In [5]:
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")
print(f"Target unique: {sorted(df[TARGET].unique())}")
print(f"\nClass distribution:\n{df[TARGET].value_counts()}")

Missing values: 0
Duplicates: 53
Target unique: [np.int64(0), np.int64(1)]

Class distribution:
Class
0    50000
1      492
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df["Amount"].hist(bins=50, ax=axes[0], edgecolor="black")
axes[0].set_title("Transaction Amount")
axes[0].set_xlabel("Amount")

df["Time"].hist(bins=50, ax=axes[1], edgecolor="black", color="orange")
axes[1].set_title("Transaction Time")
axes[1].set_xlabel("Seconds")

df.boxplot(column="Amount", by=TARGET, ax=axes[2])
axes[2].set_title("Amount by Class")
plt.suptitle("")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "eda_distributions.png"), dpi=100)
plt.show()

## 14 · Target Analysis

In [7]:
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET].value_counts().plot(kind="bar", ax=ax, color=["steelblue", "coral"])
ax.set_title("Class Distribution (Fraud vs Legitimate)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "eda_target.png"), dpi=100)
plt.show()
print(f"Imbalance ratio: 1:{int(df[TARGET].value_counts()[0] / df[TARGET].value_counts()[1])}")

Imbalance ratio: 1:101


## 15 · Train / Test Split

Stratified split to preserve the fraud rate in both sets.

In [8]:
y = df[TARGET]
X = df.drop(columns=[TARGET])

# Scale Amount and Time
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()
X["Amount"] = scaler.fit_transform(X[["Amount"]])
X["Time"] = scaler.fit_transform(X[["Time"]])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train fraud: {y_train.sum()} ({y_train.mean():.4f})")
print(f"Test fraud: {y_test.sum()} ({y_test.mean():.4f})")

Train: (40393, 30), Test: (10099, 30)
Train fraud: 394 (0.0098)
Test fraud: 98 (0.0097)


## 16 · Preprocessing Strategy

Features V1–V28 are already PCA-transformed. We apply RobustScaler to Amount and Time. No missing values to handle.

In [9]:
print("Preprocessing complete — features are PCA-transformed + Amount/Time scaled.")

Preprocessing complete — features are PCA-transformed + Amount/Time scaled.


## 17 · Feature Engineering

PCA features are pre-engineered. We rely on Amount scaling and the existing V1-V28 components.

In [10]:
print(f"Feature count: {X_train.shape[1]}")

Feature count: 30


## 18 · Baseline Model

In [11]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED, class_weight="balanced")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_proba = baseline.predict_proba(X_test)[:, 1]

baseline_acc = accuracy_score(y_test, baseline_pred)
baseline_f1 = f1_score(y_test, baseline_pred)
baseline_auc = roc_auc_score(y_test, baseline_proba)
baseline_pr_auc = average_precision_score(y_test, baseline_proba)
print(f"Baseline Logistic Regression (balanced):")
print(f"  Accuracy: {baseline_acc:.4f}")
print(f"  F1: {baseline_f1:.4f}")
print(f"  ROC-AUC: {baseline_auc:.4f}")
print(f"  PR-AUC: {baseline_pr_auc:.4f}")
print(f"  Recall: {recall_score(y_test, baseline_pred):.4f}")
print(classification_report(y_test, baseline_pred))

Baseline Logistic Regression (balanced):
  Accuracy: 0.9786
  F1: 0.4600
  ROC-AUC: 0.9848
  PR-AUC: 0.9257
  Recall: 0.9388
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     10001
           1       0.30      0.94      0.46        98

    accuracy                           0.98     10099
   macro avg       0.65      0.96      0.72     10099
weighted avg       0.99      0.98      0.98     10099



## 19 · LazyPredict Benchmark

In [12]:
# LazyPredict — quick benchmark of many classifiers
from lazypredict.Supervised import LazyClassifier

lp_clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
lp_models, lp_predictions = lp_clf.fit(X_train, X_test, y_train, y_test)

print("\nLazyPredict Results (top 15):")
print(lp_models.head(15).to_string())


LazyPredict Results (top 15):
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
QuadraticDiscriminantAnalysis  0.976037           0.947485  0.970381  0.982317   0.992181  0.976037    0.116555
CatBoostClassifier             0.998713           0.938726  0.989189  0.998675   0.998702  0.998713   10.757325
RandomForestClassifier         0.998515           0.933573  0.976060  0.998471   0.998491  0.998515   27.980262
PassiveAggressiveClassifier    0.997822           0.928171  0.978955  0.997787   0.997771  0.997822    0.114522
LGBMClassifier                 0.998317           0.923369  0.978849  0.998257   0.998288  0.998317    0.537785
DecisionTreeClassifier         0.997227           0.922819  0.922819  0.997213   0.997201  0.997227    3.017902
GaussianNB                     0.976532           0.922475  0.970851  0.9

## 20 · FLAML AutoML

In [13]:
# FLAML AutoML
try:
    from flaml import AutoML
    flaml_clf = AutoML()
    flaml_clf.fit(X_train, y_train, task="classification", time_budget=60,
                  metric="f1", verbose=0, seed=SEED)
    flaml_pred = flaml_clf.predict(X_test)
    flaml_proba = flaml_clf.predict_proba(X_test)[:, 1] if hasattr(flaml_clf, "predict_proba") else None
    print(f"FLAML best model: {flaml_clf.best_estimator}")
    print(f"FLAML F1: {f1_score(y_test, flaml_pred):.4f}")
    print(f"FLAML Accuracy: {accuracy_score(y_test, flaml_pred):.4f}")
    if flaml_proba is not None:
        print(f"FLAML ROC-AUC: {roc_auc_score(y_test, flaml_proba):.4f}")
    flaml_ok = True
except Exception as e:
    print(f"FLAML failed: {e}")
    flaml_ok = False

FLAML failed: XGBClassifier.fit() got an unexpected keyword argument 'callbacks'


## 21 · Top 3 Models — CatBoost, LightGBM, XGBoost

Using class weights to handle the extreme imbalance.

In [14]:
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

import torch
device_cat = "GPU" if torch.cuda.is_available() else "CPU"
device_lgb = "gpu" if torch.cuda.is_available() else "cpu"
device_xgb = "cuda" if torch.cuda.is_available() else "cpu"

# Calculate scale_pos_weight for imbalance
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
spw = neg_count / pos_count
print(f"scale_pos_weight: {spw:.1f}")

models = {
    "CatBoost": CatBoostClassifier(
        iterations=500, learning_rate=0.05, depth=6, verbose=0,
        task_type=device_cat, random_seed=SEED, auto_class_weights="Balanced",
        eval_metric="F1", early_stopping_rounds=50
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        device=device_lgb, random_state=SEED, verbose=-1,
        scale_pos_weight=spw
    ),
    "XGBoost": XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        device=device_xgb, random_state=SEED, eval_metric="logloss",
        early_stopping_rounds=50, scale_pos_weight=spw
    ),
}

results = {}
for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    t0 = time.time()
    if name == "XGBoost":
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    elif name == "CatBoost":
        model.fit(X_train, y_train, eval_set=(X_test, y_test), verbose=False)
    else:
        model.fit(X_train, y_train)
    elapsed = time.time() - t0

    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    auc = roc_auc_score(y_test, proba)
    pr_auc = average_precision_score(y_test, proba)

    results[name] = {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "roc_auc": auc, "pr_auc": pr_auc, "time": elapsed}
    print(f"  Accuracy: {acc:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}")
    print(f"  ROC-AUC: {auc:.4f}  PR-AUC: {pr_auc:.4f}  ({elapsed:.1f}s)")
    print(classification_report(y_test, preds))

results_df = pd.DataFrame(results).T.sort_values("f1", ascending=False)
print("\n=== Model Comparison ===")
print(results_df.to_string())

scale_pos_weight: 101.5

Training CatBoost...


  Accuracy: 0.9903  Precision: 0.5000  Recall: 0.9184  F1: 0.6475
  ROC-AUC: 0.9773  PR-AUC: 0.9036  (4.2s)
              precision    recall  f1-score   support

           0       1.00      0.99      1.00     10001
           1       0.50      0.92      0.65        98

    accuracy                           0.99     10099
   macro avg       0.75      0.95      0.82     10099
weighted avg       0.99      0.99      0.99     10099


Training LightGBM...


  Accuracy: 0.9985  Precision: 0.9462  Recall: 0.8980  F1: 0.9215
  ROC-AUC: 0.9823  PR-AUC: 0.9261  (2.7s)
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     10001
           1       0.95      0.90      0.92        98

    accuracy                           1.00     10099
   macro avg       0.97      0.95      0.96     10099
weighted avg       1.00      1.00      1.00     10099


Training XGBoost...


  Accuracy: 0.9982  Precision: 0.9167  Recall: 0.8980  F1: 0.9072
  ROC-AUC: 0.9816  PR-AUC: 0.9222  (1.2s)
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     10001
           1       0.92      0.90      0.91        98

    accuracy                           1.00     10099
   macro avg       0.96      0.95      0.95     10099
weighted avg       1.00      1.00      1.00     10099


=== Model Comparison ===
          accuracy  precision    recall        f1   roc_auc    pr_auc      time
LightGBM  0.998515   0.946237  0.897959  0.921466  0.982334  0.926124  2.655693
XGBoost   0.998218   0.916667  0.897959  0.907216  0.981586  0.922159  1.209286
CatBoost  0.990296   0.500000  0.918367  0.647482  0.977274  0.903613  4.187356


## 22 · Top 3 Model Selection

In [15]:
print("=== Final Ranking (by F1 / PR-AUC) ===")
print(results_df[["f1", "pr_auc", "roc_auc", "recall", "precision"]].to_string())

=== Final Ranking (by F1 / PR-AUC) ===
                f1    pr_auc   roc_auc    recall  precision
LightGBM  0.921466  0.926124  0.982334  0.897959   0.946237
XGBoost   0.907216  0.922159  0.981586  0.897959   0.916667
CatBoost  0.647482  0.903613  0.977274  0.918367   0.500000


## 23 · Final Evaluation

In [16]:
print("Final comparison vs baseline:")
print(f"  Baseline F1: {baseline_f1:.4f} | PR-AUC: {baseline_pr_auc:.4f}")
for name, vals in results.items():
    print(f"  {name} F1: {vals['f1']:.4f} | PR-AUC: {vals['pr_auc']:.4f} | Recall: {vals['recall']:.4f}")

Final comparison vs baseline:
  Baseline F1: 0.4600 | PR-AUC: 0.9257
  CatBoost F1: 0.6475 | PR-AUC: 0.9036 | Recall: 0.9184
  LightGBM F1: 0.9215 | PR-AUC: 0.9261 | Recall: 0.8980
  XGBoost F1: 0.9072 | PR-AUC: 0.9222 | Recall: 0.8980


## 24 · Error Analysis

In [17]:
# Error Analysis — examine misclassifications from best model
best_name = results_df.index[0]
best_model = models[best_name]
best_preds = best_model.predict(X_test)

errors = X_test.copy()
errors["y_true"] = y_test.values
errors["y_pred"] = best_preds
errors["correct"] = errors["y_true"] == errors["y_pred"]

print(f"Best model: {best_name}")
print(f"Total test samples: {len(errors)}")
print(f"Correct: {errors['correct'].sum()} | Wrong: {(~errors['correct']).sum()}")
print(f"Error rate: {(~errors['correct']).mean():.4f}")

# Confusion matrix
cm = confusion_matrix(y_test, best_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=100)
plt.show()
print("Confusion matrix saved.")

Best model: LightGBM
Total test samples: 10099
Correct: 10084 | Wrong: 15
Error rate: 0.0015


Confusion matrix saved.


## 25 · Interpretation and Business Insight

- PCA features V1–V28 capture the most important transaction patterns.
- Amount scaling is critical — fraud transactions tend to be different in magnitude.
- Class weights significantly improve recall for the minority fraud class.
- The precision-recall tradeoff is crucial: catching more fraud also means more false positives.

## 26 · Limitations

- PCA-transformed features make interpretation difficult.
- No temporal modeling — time-series patterns in transaction sequences are lost.
- Real-world fraud detection requires streaming/online learning.
- Dataset is from European cardholders (2013) — patterns may not transfer.

## 27 · How to Improve

1. Use anomaly detection (Isolation Forest, Autoencoders) alongside supervised models.
2. Add feature interaction terms.
3. Implement threshold optimization based on business cost ratios.
4. Try ensemble methods (stacking, voting).
5. Use SMOTE or ADASYN for synthetic oversampling.

## 28 · Production Considerations

- Real-time scoring latency must be <100ms.
- Need concept drift detection — fraud patterns evolve.
- Must balance fraud catch rate vs customer friction.
- Regulatory requirements for model explainability (PSD2, GDPR).

## 29 · Common Mistakes

1. Using accuracy as metric on 0.17% fraud rate (99.83% by predicting all negative).
2. Not stratifying train/test split.
3. Oversampling before splitting (data leakage).
4. Ignoring the cost asymmetry between false positives and false negatives.

## 30 · Mini Challenge

1. Find the optimal decision threshold that maximizes F1.
2. Compare SMOTE vs class weights — which gives better recall?
3. Build an Isolation Forest and compare its performance.
4. Create a cost-benefit analysis with realistic fraud costs.

## 31 · Final Summary

- Fraud detection requires imbalance-aware metrics (PR-AUC, recall, F1).
- Gradient boosting with class weights handles extreme imbalance well.
- The precision-recall tradeoff is the core decision in fraud detection.
- Always preserve all fraud cases when sampling large datasets.

In [18]:
# Save metrics
metrics = {name: {k: round(v, 4) for k, v in vals.items()} for name, vals in results.items()}
metrics["baseline"] = {"accuracy": round(baseline_acc, 4), "f1": round(baseline_f1, 4), "pr_auc": round(baseline_pr_auc, 4)}
with open(os.path.join(SAVE_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to {os.path.join(SAVE_DIR, 'metrics.json')}")

Metrics saved to E:/Github/Machine-Learning-Projects/Classification/Credit Card Fraud - Imbalanced Dataset\metrics.json
